# Day 1 — Document Ingestion
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 1 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

This notebook walks through the full Day 1 pipeline step by step: parsing a real clinical
guideline PDF, choosing a chunking strategy, generating embeddings, and building a
queryable vector index — the same steps implemented in `ingest.py`, but broken apart here
so you can inspect what happens at each stage before you rely on the script.

**By the end of this notebook you will be able to:**
1. Explain why grounding — not raw model memory — matters in clinical AI
2. Parse a PDF and inspect its extracted structure
3. Compare fixed-size vs. section-aware chunking on the same document
4. Generate an embedding and explain what the resulting vector represents
5. Build a persisted vector index and run a real query against it

> **Data source:** this notebook uses `data/WHO_Hypertension_Guideline_2021.pdf`, the real
> WHO guideline bundled with your starter kit — not a toy example.


## 0. Setup

Run this cell first. It adds the repo root to the path so we can reuse the exact same
functions defined in `ingest.py`, and confirms your environment is ready.


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config
from pathlib import Path

print("Data directory:", config.DATA_DIR)
print("Chunk size (tokens):", config.CHUNK_SIZE)
print("Chunk overlap (tokens):", config.CHUNK_OVERLAP)
print("PDFs found:", [p.name for p in config.DATA_DIR.glob("*.pdf")])


## 1. Why Grounding Matters

Before touching any code, sit with this for a second: a large language model can generate
a fluent, confident-sounding clinical recommendation **even when it has no real evidence
behind it.** It has no built-in mechanism to say "I don't know."

Retrieval-Augmented Generation (RAG) fixes this by separating two things:

- **What the model knows** (its training data — broad, but unverifiable and possibly stale)
- **What the model is allowed to say** (only what's in the text you hand it right now)

Everything you build today is the **first half** of that separation: turning a trustworthy
PDF into a searchable, citable index. Day 3 builds the second half (forcing the model to
answer only from what this index returns).


## 2. Step 1 — Parse the PDF

`PyPDFLoader` reads a PDF and returns one LangChain `Document` per page, each carrying
page-level metadata automatically (page number, source path).

Run the cell below and inspect the output. Notice that `page.metadata["page"]` is
**zero-indexed** — page index `0` is the PDF's first page.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(config.DATA_DIR.glob("*.pdf"))[0]
print(f"Loading: {pdf_path.name}\n")

loader = PyPDFLoader(str(pdf_path))
raw_pages = loader.load()

print(f"Loaded {len(raw_pages)} pages.\n")
print("--- Page 3 (index 2) raw metadata, as PyPDFLoader gives it to us ---")
print(raw_pages[2].metadata)
print("\n--- Page 3 (index 2) first 400 characters ---")
print(raw_pages[2].page_content[:400])


### Raw metadata isn't citation-ready yet

Look at the metadata above: PyPDFLoader gives you a generic `page` index and a full file
`source` path — but nothing called `document_name`, and no human-friendly 1-indexed page
number. If we chunk these pages as-is, every citation later would say "unknown, page ?".

`load_pdfs()` in `ingest.py` does one small but critical thing: it stamps
`document_name` and a 1-indexed `page_number` onto every page's metadata **before**
chunking, so that metadata survives all the way through to the final citation.


In [ ]:
from ingest import load_pdfs

pages = load_pdfs(config.DATA_DIR)

print("--- Page 3 (index 2) metadata AFTER normalization ---")
print({k: pages[2].metadata[k] for k in ["document_name", "page_number", "page"]})


### Checkpoint 1

Look at the printed text above. Answer for yourself before moving on:

- Are section headings ("3.1 Blood pressure threshold...") visible as recognizable text, or
  did they get mangled?
- Are there any obvious parsing artifacts (broken words, merged columns, stray characters)?

If parsing looks clean here, section-aware chunking (Step 2) will work well. If it looks
messy, no chunking strategy will fully save you — the fix belongs upstream, in parsing.


## 3. Step 2 — Compare Chunking Strategies

We'll build **two** chunkers on the same pages and compare them directly: a naive
fixed-size splitter, and the section-aware splitter actually used in `ingest.py`.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Naive fixed-size splitter: no regard for sentence/paragraph boundaries ---
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # characters, not tokens — deliberately crude
    chunk_overlap=0,
    separators=[""],       # forces raw character-count splitting
)
naive_chunks = naive_splitter.split_documents(pages)

# --- Section-aware splitter: same one used in ingest.py ---
aware_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config.CHUNK_SIZE * 4,      # ~4 chars/token estimate
    chunk_overlap=config.CHUNK_OVERLAP * 4,
    separators=["\n\n", "\n", ". ", " ", ""],
)
aware_chunks = aware_splitter.split_documents(pages)

print(f"Naive fixed-size chunker:   {len(naive_chunks)} chunks")
print(f"Section-aware chunker:     {len(aware_chunks)} chunks")


In [ ]:
# Look at one naive chunk boundary — notice it can cut mid-sentence
print("--- Naive chunk #5 (often cuts mid-sentence) ---")
print(repr(naive_chunks[5].page_content))

print("\n--- Section-aware chunk #5 (respects paragraph breaks) ---")
print(repr(aware_chunks[5].page_content[:300]))


### Checkpoint 2

Compare the two printed chunks above.

- Does the naive chunk end mid-word or mid-sentence?
- Does the section-aware chunk end at a more natural paragraph or sentence boundary?

This is the entire argument for section-aware chunking in one comparison: **the boundary
you cut at becomes the boundary a citation has to point to.** A citation that lands mid-sentence
is much harder for a clinician to trust and verify.


## 4. Step 3 — Attach Citation Metadata

A chunk without a traceable source is useless for a clinical tool. Before embedding
anything, every chunk needs: **document name, page number, and a stable chunk id.**
This is exactly what `chunk_documents()` in `ingest.py` does — let's call it directly.


In [ ]:
from ingest import chunk_documents

chunks = chunk_documents(pages)
print(f"Total chunks with metadata attached: {len(chunks)}\n")

sample = chunks[10]
print("--- Sample chunk metadata ---")
for k in ["document_name", "page_number", "chunk_id"]:
    print(f"  {k}: {sample.metadata.get(k)}")
print("\n--- Sample chunk text ---")
print(sample.page_content[:300])


## 5. Step 4 — What Is an Embedding, Really?

An embedding model converts text into a list of numbers (a **vector**) that captures
meaning — texts about similar topics end up as vectors that point in similar directions,
even if they don't share any of the same words.

Let's embed three short phrases and check which two are "closer" in vector space.


In [ ]:
import numpy as np
from ingest import get_embedding_function

embed_fn = get_embedding_function()

texts = [
    "first-line treatment for hypertension",
    "initial therapy for high blood pressure",   # means the same thing, different words
    "recommended screening interval for breast cancer",  # unrelated topic
]

vectors = embed_fn.embed_documents(texts)
vectors = np.array(vectors)
print(f"Each embedding is a vector of length {vectors.shape[1]}\n")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_related = cosine_similarity(vectors[0], vectors[1])
sim_unrelated = cosine_similarity(vectors[0], vectors[2])

print(f"Similarity — same meaning, different words:  {sim_related:.3f}")
print(f"Similarity — genuinely different topics:      {sim_unrelated:.3f}")


### Checkpoint 3

You should see the first similarity score noticeably **higher** than the second — the
"same meaning, different words" pair should score closer together, even though they share
almost no words in common. This is the entire mechanism semantic search relies on. If both
scores came out similar, something about the embedding model would be worth investigating
before trusting it on Day 2.


## 6. Step 5 — Build the Vector Index

Now we embed every chunk and store it in a local ChromaDB collection, using the exact
same `build_index()` function from `ingest.py`. This is the same call the script makes —
seeing it run here just makes the process visible.

> First run downloads a small local embedding model (~100MB) — this happens once and is
> cached afterward.


In [ ]:
from ingest import build_index

vectordb = build_index(chunks)
print("\nIndex build complete.")


## 7. Step 6 — Run a Real Query

The whole point of everything above: ask a real clinical question and see whether the
index returns something relevant.


In [ ]:
question = "What is the target blood pressure for a patient with cardiovascular disease?"

results = vectordb.similarity_search_with_relevance_scores(question, k=3)

print(f"Question: {question}\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"[{i}] score={score:.3f}  {doc.metadata.get('document_name')}, "
          f"page {doc.metadata.get('page_number')}")
    print(f"    \"{doc.page_content[:180].strip()}...\"\n")


### Checkpoint 4 — Day 1 Self-Check

Before you close this notebook, confirm all of the following are true:

- [ ] The top retrieved chunk in Step 6 is genuinely relevant to the question asked
- [ ] Every result shows a document name and page number (not `None`)
- [ ] You could explain to a teammate, in one sentence, why section-aware chunking beat
      the naive splitter in Checkpoint 2

If any of these aren't true yet, that's normal — go back to the relevant step above and
adjust `config.py` (chunk size, overlap) before moving on to Day 2.

## What's Next

Day 2's notebook picks up exactly here: tuning `top_k`, benchmarking this embedding model
against alternatives, and proving your retrieval quality with real, logged numbers instead
of a single example query.
